Imports:

In [1]:
import json
import os
from bs4 import BeautifulSoup
from requests import RequestException
import re
import wikipediaapi
import requests

In [2]:
USER_AGENT = os.getenv('Learning project: TourGuideAI/1.0', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
                          ' Chrome/91.0.4472.124 Safari/537.36')

JSON-Dateien aus dem Verzeichnis 'data' laden und in einer Liste speichern:

In [3]:
def load_data_von_DataVerzeichnis(path='../data'):
    data = []
    for filename in os.listdir(path):
        # Nur JSON-Dateien verarbeiten
        if filename.endswith('.json'):
            # Öffnen und Laden der JSON-Datei im Lesemodus
            with open(os.path.join(path, filename), 'r', encoding='utf-8') as f:
                # Inhalt der JSON-Datei laden
                content = json.load(f)
                if isinstance(content, list):
                    data.extend(content)
                else:
                    data.append(content)
    return data

data = load_data_von_DataVerzeichnis()
print(f'{len(data)} documents loaded.')

13 documents loaded.


In [4]:
urls = [
    'https://www.museum-barberini.de/de/',
    'https://shop.museum-barberini.de/#/tickets/time?group=timeSlot&lang=de',
    'https://shop.museum-barberini.de/#/tickets/combi'
    'https://shop.museum-barberini.de/#/tickets/time?group=timeSlot',
    'https://www.museum-barberini.de/de/kalender/',
    'https://www.industriemuseum-brandenburg.de/home'
]

In [5]:
# HTML von einer URL laden und in BeautifulSoup-Objekt umwandeln
def get_soup(url):
    headers = {"Accept": "text/html,application/json",
               'User-Agent': USER_AGENT}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status() #wenn !200, Fehler werfen
    return BeautifulSoup(response.text, 'html.parser')

def extract_opening_hours(soup):
    results = []
    for text in soup.find_all(string=True):
        if text and "uhr" in text.lower():
            cleaned = text.strip() # Whitespace entfernen
            if 5 < len(cleaned) < 200: # Nur relevante Texte behalten
                results.append(cleaned)
    return list(set(results))


def extract_ticket_prices(soup):
    results = []
    price_pattern = re.compile(r"\d+[.,]?\d*\s?€")

    for text in soup.find_all(string=True):
        if text:
            matches = price_pattern.findall(text)
            results.extend(matches)

    return list(set(results))


def extract_events_by_date(soup, date_string):
    events = []
    for element in soup.find_all(["h2", "h3", "p", "li"]):
        text = element.get_text(strip=True)
        if date_string in text:
            events.append(text)
    return events


def process_urls(urls, date_string):
    results = {}

    for url in urls:
        try:
            soup = get_soup(url)
            data = {
                "Öffnungszeiten": extract_opening_hours(soup),
                "Tickets": extract_ticket_prices(soup),
                "Events": extract_events_by_date(soup, date_string)
            }
            results[url] = data

        except Exception as e:
            print(f"Fehler bei {url}: {e}")

    return results


results = process_urls(urls, "12.04.2026")

print(json.dumps(results, indent=4, ensure_ascii=False))

{
    "https://www.museum-barberini.de/de/": {
        "Öffnungszeiten": [
            "Mi,  4. März, 15 Uhr\t                        und weitere Termine",
            "Fr, 20. März, 19 Uhr",
            "Heute bis 19 Uhr",
            "Mi,  4. März, 16 Uhr\t                        und weitere Termine",
            "Do,  5. März, 17 Uhr",
            "Do,  5. März, 13 Uhr\t                        und weitere Termine",
            "Do, 5. März, 19 Uhr"
        ],
        "Tickets": [],
        "Events": []
    },
    "https://shop.museum-barberini.de/#/tickets/time?group=timeSlot&lang=de": {
        "Öffnungszeiten": [],
        "Tickets": [],
        "Events": []
    },
    "https://shop.museum-barberini.de/#/tickets/combihttps://shop.museum-barberini.de/#/tickets/time?group=timeSlot": {
        "Öffnungszeiten": [],
        "Tickets": [],
        "Events": []
    },
    "https://www.museum-barberini.de/de/kalender/": {
        "Öffnungszeiten": [
            "14 Uhr",
            "16 

In [6]:
import requests
import json
import os

USER_AGENT = os.getenv("USER_AGENT", "Mozilla/5.0 (Windows NT 10.0; Win64; x64)")

def get_ticket_data(date):
    url = (
        "https://barberini.gomus.de/api/v4/tickets"
        "?by_bookable=true"
        "&by_free_timing=false"
        "&by_ticket_type=time_slot"
        "&locale=de"
        "&per_page=1000"
        f"&valid_at={date}"
    )

    headers = {
        "Accept": "application/json",
        "x-shop-url": "shop.museum-barberini.de",
        "User-Agent": USER_AGENT
    }

    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return response.json()

def extract_prices(data):
    results = []
    for ticket in data.get("tickets", []):

        results.append({
            "name": ticket.get("title"),
            "price in EUR": ticket.get("price_cents", 0) / 100,
            "ticket_type": ticket.get("ticket_type"),
            "free_timing": ticket.get("free_timing"),
            "bookable": ticket.get("bookable")
        })
    return results

# Beispiel: 5. März 2026
date = "2026-03-05"
data = get_ticket_data(date)
prices = extract_prices(data)

print(json.dumps(prices, indent=2, ensure_ascii=False))

[
  {
    "name": "Hausticket regulär",
    "price in EUR": 16.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket ermäßigt",
    "price in EUR": 10.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket frei (0 € Ticket)",
    "price in EUR": 0.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket Barberini am Abend",
    "price in EUR": 10.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket frei Barberini am Abend",
    "price in EUR": 0.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket regulär ",
    "price in EUR": 18.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket ermäßigt",
    "price in EUR": 10.0,
    "tic

URLs lesen und Texte aus den Dokumenten extrahieren:

In [7]:
# HTML ohne Webloader laden

def load_html(url: str) -> BeautifulSoup:
    headers = {'User-Agent': os.getenv('USER_AGENT')}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except RequestException as e:
        print(f"Fehler beim Laden von {url}: {e}")
        return None
    return BeautifulSoup(response.text, 'html.parser')

In [8]:
# Regex Patterns
ADDRESS_REGEX = re.compile(
    r"\b([A-ZÄÖÜ][\wäöüß\-]+(?:straße|strasse|weg|allee|platz|gasse|ring|damm|ufer|hof|steig|chaussee|promenade|markt|pfad))"
    r"\s+(\d+[a-zA-Z]?)"
    r"\s+(\d{5})"
    r"\s+([A-ZÄÖÜ][a-zäöüß]+(?:\s+[a-zäöüß]+)*(?:\s+[A-ZÄÖÜ][a-zäöüß]+)*)"
    r"(?=\s*(?:$|<|\.|,|;|:|\n|\bTelefon\b|\bTel\b|\bFax\b|\bE-Mail\b|\bEmail\b|\bÖffnungszeiten\b))",
    flags=re.IGNORECASE
)
ADDRESS_REGEX = re.compile(
    r"([A-ZÄÖÜ][\wäöüß\-]+\s*(?:straße|strasse|weg|allee|platz|gasse|ring|damm|ufer|hof|steig|chaussee|promenade|markt|pfad)?)\s+(\d+[a-zA-Z]?)?,?\s*(\d{5})?\s*([A-ZÄÖÜ][a-zäöüß]+(?:\s+[a-zäöüß]+)*)?",
    flags=re.IGNORECASE
)

STUNDEN_REGEX = re.compile(r"\b(?:Mo|Di|Mi|Do|Fr|Sa|So)\.?\s*(?:\d{1,2}:\d{2}\s*-\s*\d{1,2}:\d{2}|geschlossen)\b",  flags=re.IGNORECASE)
PREIS_REGEX = re.compile(r"\b(?:Eintritt|Preis|Kosten|Tarif|Gebühr).{0,20}?\b\d{1,3}(?:,\d{2})?\s*€\b",  flags=re.IGNORECASE)

In [9]:
# async def load_html_js(url: str) -> BeautifulSoup:
#     async with async_playwright() as p:
#         browser = await p.chromium.launch(headless=True)
#         page = await browser.new_page()
#         await page.goto(url, timeout=30000)
#         await page.wait_for_load_state('networkidle')
#
#         html = await page.evaluate("""
#                                    () => {
#                                        function extract(el) {
#                                            let text = el.innerText || "";
#                                            if (el.shadowRoot) {
#                                                text += " " + el.shadowRoot.innerText;
#                                                el.shadowRoot.querySelectorAll("*").forEach(child => {
#                                                    text += " " + extract(child);
#                                                });
#                                            }
#                                            return text;
#                                        }
#
#                                        let full = document.body.innerText + " ";
#                                        document.querySelectorAll("*").forEach(el => {
#                                            full += " " + extract(el);
#                                        });
#
#                                        return full;
#                                    }
#                                    """)
#
#         await browser.close()
#
#     return BeautifulSoup(html, "lxml")


In [10]:
# Facts extrahieren

def extract_facts(soup: BeautifulSoup) -> dict:
    if soup is None:
        return {}

    text = ' '.join(soup.get_text(' ').split())
    facts = {}

    adresse = ADDRESS_REGEX.search(text)
    stunden = STUNDEN_REGEX.search(text)
    preis = PREIS_REGEX.search(text)

    if adresse:
        facts['Adresse'] = adresse.group(0)
    if stunden:
        facts['Eintritt'] = stunden.group(0)
    if preis:
        facts['Preis'] = preis.group(0)

    return facts


In [12]:
urlWiki = [
    'https://de.wikipedia.org/wiki/Brandenburger_Tor',
    'https://de.wikipedia.org/wiki/Schloss_Brandenburg_an_der_Havel',
    'https://de.wikipedia.org/wiki/Brandenburger_Dom',
    'https://de.wikipedia.org/wiki/Brandenburger_Tor_(Potsdam)',

]
wiki = wikipediaapi.Wikipedia(language='de',
                             extract_format=wikipediaapi.ExtractFormat.WIKI,
                             user_agent=USER_AGENT)

def load_wikipedia_text(title: str) -> dict:
    page = wiki.page(title)
    if not page.exists():
        return {'error': 'Page not found'}
    text = page.text
    return {
        'text': text,
        'title': page.title,
        'source': page.fullurl,
        'license': 'CC BY-SA 4.0'
    }

#Titel aus URL extrahieren
def extract_title_from_url(url: str) -> str:
    match = re.search(r'/wiki/([^#]+)', url)
    if match:
        return match.group(1)
    return None

wiki_data = {}
for url in urlWiki:
    title = extract_title_from_url(url)
    wiki_data[title] = load_wikipedia_text(title)

# Ergebnis als JSON ausgeben
import json
print(json.dumps(wiki_data, indent=2, ensure_ascii=False))

{
  "Brandenburger_Tor": {
    "text": "Das Brandenburger Tor in Berlin ist ein frühklassizistisches Triumphtor, das an der Westflanke des quadratischen Pariser Platzes im Berliner Ortsteil Mitte steht. Es wurde als Abschluss der zentralen Prachtstraße der Dorotheenstadt, des Boulevards Unter den Linden, in den Jahren 1789 bis 1793 auf Anweisung des preußischen Königs Friedrich Wilhelm II. nach Entwürfen von Carl Gotthard Langhans errichtet. Die das Tor krönende Skulptur der Quadriga ist ein Werk nach dem Entwurf des Bildhauers Johann Gottfried Schadow. Westlich des Brandenburger Tores befinden sich die ausgedehnten Grünflächen des Großen Tiergartens, die in gradliniger Verlängerung der Straße Unter den Linden von der Straße des 17. Juni durchquert werden. Die Platzfläche unmittelbar westlich des Tores trägt den Namen Platz des 18. März.\nDas Tor ist das einzig erhaltene von zuletzt 18 Berliner Stadttoren. In der Formensprache stellt es die Hinwendung vom römischen zum griechischen Vor